# What this one does is call upon a windowing method that appends to the start/end of the word. 

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from run_spike_pipeline import run_patient_pipeline
from build_cleanXY import build_cleanX_by_speaker_condition, clean_inputs, build_cleanX_from_spike_dict
from sweep_regression import sweep_all_configs

def sweep_windows_over_patient(patient_id, patient_prefix,
                               region="hippocampus",
                               window_sweeps=None,
                               test_sizes=[0.3],
                               n_pcs_list=[30],
                               alpha_grid=[0.01, 0.1, 1.0],
                               n_shuffles=20,
                               n_jobs=4,
                               spike_base_dir="/Users/aniluchavez/Documents/Language/Python/spikesforw2v"):

    mat_base = "/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/YFK" ### CHANGE HERE FOR EACH PATIENT
    excel_base = "/Users/aniluchavez/Documents/Language/Python/final_word2vec"
    embedding_csv = f"{excel_base}/{patient_prefix}_words_word2vec/{patient_prefix}_aligned_word2vec_embeddings.csv"

    if window_sweeps is None:
        # Load the Excel just once to detect active speakers
        excel_path = os.path.join(excel_base, f"{patient_prefix}_filtered_used_rows_word2vec.xlsx")
        df = pd.read_excel(excel_path)
        speaker_columns = [col for col in df.columns if str(col).lower().startswith("speaker")]

        print(f"🧠 Found speaker columns in Excel: {speaker_columns}")

        # Dynamically build window_sweeps for all conditions
        window_sweeps = {}
        for pre in [2000]:
            for post in [0]:
                name = f"self_plusminus_{pre}_{post}"
                config = {}

                for col in speaker_columns:
                    if col == "Speaker1":
                        config[col] = {"mode": "onset_to_offset_plusminus", "pre": pre, "post": post}
                    else:
                        config[col] = {"mode": "onset_to_offset_plus", "post": 80}

                window_sweeps[name] = config


    for config_name, speaker_window_modes in window_sweeps.items():
        print(f"\n=== 🪟 Running window config: {config_name} ===")
        output_suffix = f"w2v_{config_name}"

        run_patient_pipeline(
            patient_id=patient_id,
            patient_prefix=patient_prefix,
            mat_base=mat_base,
            excel_base=excel_base,
            region_ranges={
                                    "hippocampus": [(1,16),(25,40)],#,,(25,40)
                                    "ACC": [(49,56)],#(41,48),
                                    "thalamus":[(17,24)],
                                     "OFC":[(41,48)],### CHANGE REGIONS HERE FOR EACH PATIENT
                    },
            speaker_window_modes=speaker_window_modes,
            binSize=20,
            output_suffix=output_suffix,
            spike_base_dir=spike_base_dir
        )

        spike_dir = os.path.join(spike_base_dir, f"output_{patient_prefix}_english_only_{output_suffix}")
        meta_xlsx = os.path.join(spike_dir, f"{patient_id}_with_regress_dur_{output_suffix}.xlsx")

        if not os.path.exists(meta_xlsx):
            raise FileNotFoundError(f"\ud83d\udea8 Expected Excel file not found: {meta_xlsx}")

        (X_self, Y_self, dur_self), (X_other, Y_other, dur_other) = build_cleanX_by_speaker_condition(
            embedding_csv, meta_xlsx, spike_dir,
            region=region,
            target_speaker="SPK1",
            embedding_col="Embedding",
            separate_self_other=True
        )
        print(f"X_self.shape: {X_self.shape}, Y_self.shape: {Y_self.shape}, dur_self.shape: {dur_self.shape}")
        print(f"X_other.shape: {X_other.shape}, Y_other.shape: {Y_other.shape}, dur_other.shape: {dur_other.shape}")

        X_self, Y_self, dur_self = clean_inputs(X_self, Y_self, dur_self)
        X_other, Y_other, dur_other = clean_inputs(X_other, Y_other, dur_other)

        if len(X_self) > 0 and Y_self.shape[1] > 0:
            print("Running regression on SELF...")
            df_self = sweep_all_configs(
                X_embed=X_self,
                dur=dur_self,
                Y=Y_self,
                test_sizes=test_sizes,
                n_pcs_list=n_pcs_list,
                alpha_grid=alpha_grid,
                n_shuffles=n_shuffles,
                n_jobs=n_jobs
            )
            df_self["condition"] = "self"
            df_self["window_config"] = config_name
        else:
            df_self = pd.DataFrame()

        if len(X_other) > 0 and Y_other.shape[1] > 0:
            print("\ud83d\udd01 Running regression on OTHER...")
            df_other = sweep_all_configs(
                X_embed=X_other,
                dur=dur_other,
                Y=Y_other,
                test_sizes=test_sizes,
                n_pcs_list=n_pcs_list,
                alpha_grid=alpha_grid,
                n_shuffles=n_shuffles,
                n_jobs=n_jobs
            )
            df_other["condition"] = "other"
            df_other["window_config"] = config_name
        else:
            df_other = pd.DataFrame()

        df_combined = pd.concat([df_self, df_other], ignore_index=True)
        output_csv = f"regression_results_{patient_prefix}_{config_name}.csv"
        df_combined.to_csv(output_csv, index=False)
        print(f" Saved regression results to {output_csv}")



# This method actually does a much smoother way by doing a shifting window with that window being constant being the word duration (Note that this method has to be run patient by patient)

In [ ]:
def sweep_shifts_over_patient(patient_id, patient_prefix,
                              regions=["hippocampus"],
                              region_ranges=None,
                              shift_range=range(-1000, 1001, 50),
                              test_sizes=[0.3],
                              n_pcs_list=[30],
                              alpha_grid=[0.01, 0.1, 1.0],
                              n_shuffles=20,
                              n_jobs=4,
                              spike_base_dir="/Users/aniluchavez/Documents/Language/Python/spikesforw2v"):
    from run_spike_pipeline import run_patient_pipeline
    from build_cleanXY import build_cleanX_from_spike_dict, clean_inputs
    from sweep_regression import sweep_all_configs
    import os
    import pandas as pd

    subfolder = patient_id.split("_")[0].replace("pt", "")
    mat_base = f"/Users/aniluchavez/Documents/MATLAB/Language/PatientData/SpikesMAT/{subfolder}"
    excel_base = "/Users/aniluchavez/Documents/Language/Python/final_word2vecW"
    embedding_csv = f"{excel_base}/{patient_prefix}_words_word2vec/{patient_prefix}_aligned_word2vec_embeddings.csv"
    base_excel = os.path.join(excel_base, f"{patient_prefix}_words_word2vec", f"{patient_prefix}_filtered_used_rows_word2vec.xlsx")

    df = pd.read_excel(base_excel)
    speaker_columns = [col for col in df.columns if str(col).lower().startswith("speaker")]
    print(f"🧠 Found speaker columns: {speaker_columns}")

    window_sweep_config = {
        spk: {"mode": "onset_to_offset"} for spk in speaker_columns
    }

    all_results = []

    for shift in shift_range:
        print(f"\n=== 🪟 Running shift {shift:+} ms ===")
        output_suffix = f"shift{shift:+d}ms".replace("+", "p").replace("-", "m")

        spike_data, spike_dir, meta_xlsx = run_patient_pipeline(
            patient_id=patient_id,
            patient_prefix=patient_prefix,
            mat_base=mat_base,
            excel_base=excel_base,
            region_ranges=region_ranges,
            speaker_window_modes=window_sweep_config,
            output_suffix=output_suffix,
            spike_base_dir=spike_base_dir,
            use_sliding_window=True,
            shift_ms=shift,
            return_spike_data=True
        )

        for region in regions:
            print(f"🧠 Region: {region}")

            (X_self, Y_self, dur_self), (X_other, Y_other, dur_other) = build_cleanX_from_spike_dict(
                embedding_csv, meta_xlsx, spike_data,
                region=region,
                target_speaker="SPK1",
                embedding_col="Embedding",
                separate_self_other=True
            )

            X_self, Y_self, dur_self = clean_inputs(X_self, Y_self, dur_self)
            X_other, Y_other, dur_other = clean_inputs(X_other, Y_other, dur_other)

            def run_condition(X, Y, dur, label):
                if len(X) > 0 and Y.shape[1] > 0:
                    df = sweep_all_configs(
                        X_embed=X, dur=dur, Y=Y,
                        test_sizes=test_sizes,
                        n_pcs_list=n_pcs_list,
                        alpha_grid=alpha_grid,
                        n_shuffles=n_shuffles,
                        n_jobs=n_jobs
                    )
                    df["condition"] = label
                    df["shift_ms"] = shift
                    df["region"] = region
                    return df
                return pd.DataFrame()

            df_self = run_condition(X_self, Y_self, dur_self, "self")
            df_other = run_condition(X_other, Y_other, dur_other, "other")

            all_results.append(df_self)
            all_results.append(df_other)

    df_all = pd.concat(all_results, ignore_index=True)
    output_csv = f"regression_results_{patient_prefix}_ALLREGIONS_shifts.csv"
    df_all.to_csv(output_csv, index=False)
    print(f"✅ Saved all-region shift sweep to: {output_csv}")
    return df_all

def run_shift_sweep_all_patients(patient_configs,
                                 shift_range=range(-1000, 1001, 10),
                                 test_sizes=[None],
                                 n_pcs_list=[100],
                                 alpha_grid=None,
                                 n_shuffles=20,
                                 n_jobs=-1,
                                 spike_base_dir="/Users/aniluchavez/Documents/Language/Python/spikesforw2v"):
    """
    Run sweep_shifts_over_patient for multiple patients using custom region mappings.
    """
    for config in patient_configs:
        patient_id = config["patient_id"]
        patient_prefix = config["patient_prefix"]
        region_map = config["region_ranges"]
        regions = list(region_map.keys())

        print(f"\n🔁 Running patient: {patient_prefix} | Regions: {regions}")
        
        sweep_shifts_over_patient(
            patient_id=patient_id,
            patient_prefix=patient_prefix,
            regions=regions,
            region_ranges=region_map,
            shift_range=shift_range,
            test_sizes=test_sizes,
            n_pcs_list=n_pcs_list,
            alpha_grid=alpha_grid,
            n_shuffles=n_shuffles,
            n_jobs=n_jobs,
            spike_base_dir=spike_base_dir,
        )




# This runs one patient

In [ ]:
shift_range = range(-1000, 1001, 10)  
import numpy as np
import numpy as np

alpha_grid_final = np.unique(np.concatenate([
    [0.1],                                # manual low fallback
    np.logspace(0, 2.5, 10),              # your original main sweep: 1 → ~316
    [1000.0],                             # manual high fallback
    np.logspace(-2, 4, 20)               # new full-range sweep: 0.01 → 10000
]))

sweep_shifts_over_patient(
    patient_id="ptYFF_task17",
    patient_prefix="PTYFF_task17",
    regions=["hippocampus", "ACC"],
    shift_range=range(-1000, 1001, 10),
    test_sizes=[None],
    n_pcs_list=[100],
    alpha_grid=alpha_grid_final,
    n_shuffles=20,
    n_jobs=-1,
)




# This runs multipatient

In [ ]:
patient_configs = [
        # {
        #     "patient_id": "ptYEY_task86",
        #     "patient_prefix": "PTYEY_task86",
        #     "region_ranges": {
        #         "hippocampus": [(1, 16)],
        #     }
        # },

        {
            "patient_id": "ptYFC_task28",
            "patient_prefix": "PTYFC_task28",
            "region_ranges": {
                "hippocampus": [(1,8),(33,48)],
                "ACC": [(17,32),(49,64)]
            }
        },
        {
            "patient_id": "ptYFI_task81",
            "patient_prefix": "PTYFI_task81",
            "region_ranges": {
                "hippocampus": [(1,8),(25,40)],
                "ACC": [(9,16)]
            }
        },
            {
            "patient_id": "ptYEU_task147",
            "patient_prefix": "PTYEU_task147",
            "region_ranges": {
                "hippocampus": [(1,16),(25,40)],
                "ACC": [(17,24),(41,48)]
            }
        },
        {
            "patient_id": "ptYFG_task18",
            "patient_prefix": "PTYFG_task18",
            "region_ranges": {
                "hippocampus": [(1,8),(33,48)],
                "ACC": [(17,32),(49,64)]
            }
        },
        {
            "patient_id": "ptYEZ_task60",
            "patient_prefix": "PTYEZ_task60",
            "region_ranges": {
                "hippocampus": [(1,16)],
                "ACC": [(17,24)]
            }
        },
        {
            "patient_id": "ptYFA_task25",
            "patient_prefix": "PTYFA_task25",
            "region_ranges": {
                "hippocampus": [(1,16),(25,40)],
                "ACC": [(17,24)],
            }
        },
        {
            "patient_id": "ptYEV_task37",
            "patient_prefix": "PTYEV_task37",
            "region_ranges": {
                "hippocampus": [(1,16),(25,40)],
                "ACC": [(17,24),(41,48)]
            }
         },
            {
            "patient_id": "ptYFK_task40",
            "patient_prefix": "PTYFK_task40",
            "region_ranges": {
                "hippocampus": [(1,16),(25,40)],
                "ACC": [(49,56)],
            }
        },

        {
            "patient_id": "ptYFF_task17",
            "patient_prefix": "PTYFF_task17",
            "region_ranges": {
                "hippocampus": [(9,16),(25,40)],
                "ACC": [(17,24),(41,48)]
            }
        },
        # # Add more patients as needed
    ]
alpha_grid_final = np.unique(np.concatenate([
    [0.1],
    np.logspace(0, 2.5, 10),
    [1000.0],
    np.logspace(-2, 4, 20)
]))

run_shift_sweep_all_patients(
    patient_configs=patient_configs,
    shift_range=range(-1000, 1001, 10),
    test_sizes=[None],
    n_pcs_list=[100],
    alpha_grid=alpha_grid_final,
    n_shuffles=20,
    n_jobs=-1,
)


#vISUALS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the results
# df = pd.read_csv("/Users/aniluchavez/Documents/Language/Python/regression_results_PTYFC_task28_shifts.csv")

# Load your results
df = pd.read_csv("/Users/aniluchavez/Documents/Language/Python/regression_results_PTYEY_task86_ALLREGIONS_shifts.csv")

# Get unique regions
regions = df["region"].unique()

# Loop through each region and plot separately
for region in regions:
    df_region = df[df["region"] == region]

    plt.figure(figsize=(10, 5))
    sns.lineplot(
        data=df_region,
        x="shift_ms",
        y="ll_diff",
        hue="condition",
        errorbar="ci",   # 95% CI by default
        marker="o"
    )
    plt.axvline(0, color='gray', linestyle='--', label="Word onset")
    plt.title(f"LLH Diff ± 95% CI vs. Shift — {region}")
    plt.xlabel("Shift (ms)")
    plt.ylabel("LLH Diff")
    plt.grid(False)
    plt.tight_layout()
    plt.show()



# tHIs is part to define windows and alpha grid

In [ ]:
import numpy as np

alpha_grid_final = np.unique(np.concatenate([
    [0.1],                        # low fallback
    np.logspace(0, 2.5, 10),      # main sweep: 1 → ~316
    [1000.0]                      # high fallback
]))
# === Self plusminus sweeps ===

# pre_vals = [0,50]
# post_vals = [0]

# window_sweeps = {}
# for pre in pre_vals:
#     for post in post_vals:
#         label = f"self_plusminus_{pre}_{post}"
#         window_sweeps[label] = {
#             "Speaker1": {"mode": "onset_to_offset_plusminus", "pre": pre, "post": post},
#             **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
#         }

# # === Other sweeps ===
# # Define your sweep values
# other_pre_vals = [0]
# other_post_vals = [0,50]

# # Create the dictionary
# other_sweeps = {}

# for pre in other_pre_vals:
#     for post in other_post_vals:
#         label = f"other_plusminus_{pre}_{post}"
#         other_sweeps[label] = {
#             "Speaker1": {"mode": "onset_to_offset_plus", "post": 80},  # optional: fix or skip
#             **{
#                 f"Speaker{i}": {"mode": "onset_to_offset_plusminus", "pre": pre, "post": post}
#                 for i in range(2, 9)
#             }
#         }

# # Merge into the main sweep config
# window_sweeps.update(other_sweeps)





# window_sweeps = {}

# # Sweep Speaker1 only
# for pre in [5, 10, 20, 150]:
#     for post in [0, 30]:
#         label = f"self_plusminus_{pre}_{post}"
#         window_sweeps[label] = {
#             "Speaker1": {"mode": "onset_to_offset_plusminus", "pre": pre, "post": post}
#         }

# # Sweep Speaker2 only
# for pre in [5,10]:
#     for post in [0, 50,80,200,500]:
#         label = f"Speaker2_plusminus_{pre}_{post}"
#         window_sweeps[label] = {
#             f"Speaker{i}": {"mode": "onset_to_offset_plusminus", "pre": pre, "post": post}
#             for i in range(2, 9)
#         }



window_sweeps = {}

# Case 1: All speakers with 0 pre and 0 post
label = "all_zero_zero"
window_sweeps[label] = {
    f"Speaker{i}": {"mode": "onset_to_offset_plusminus", "pre": 0, "post": 0}
    for i in range(1, 9)
}

# Case 2: Speaker1 gets 50 pre, 0 post; others get 0 pre, 50 post
label = "speaker1_pre50__others_post50"
window_sweeps[label] = {
    "Speaker1": {"mode": "onset_to_offset_plusminus", "pre": 50, "post": 0},
    **{
        f"Speaker{i}": {"mode": "onset_to_offset_plusminus", "pre": 0, "post": 50}
        for i in range(2, 9)
    }
}


In [ ]:
sweep_windows_over_patient(
    patient_id="ptYFK_task40",
    patient_prefix="PTYFK_task40",
    alpha_grid=alpha_grid_final,
    n_pcs_list=[100],
    test_sizes=[None],
    n_jobs=-1,
    spike_base_dir="/Users/aniluchavez/Documents/Language/Python/spikesforw2v",
    window_sweeps=window_sweeps
)

In [ ]:
import os
import pandas as pd

# === Set base directory ===
base_dir = "/Users/aniluchavez/Documents/Language/Python"  # change this if running locally
all_files = os.listdir(base_dir)

# === Split by type ===
self_files = [f for f in all_files if f.startswith("regression_results_PTYFC_task28_self_plusminus_")]
other_files = [f for f in all_files if f.startswith("regression_results_PTYFC_task28_other_plusminus_")]

# === Define helper ===
def compute_median_ll_diff(files, label):
    records = []
    for fname in files:
        fpath = os.path.join(base_dir, fname)
        df = pd.read_csv(fpath)

        # Extract pre/post from filename
        tag = fname.split("plusminus_")[1].replace(".csv", "")
        pre, post = map(int, tag.split("_"))
        window_size = pre + post

        # Compute median ll_diff
        median_ll = df["ll_diff"].median()

        records.append({
            "source": label,
            "filename": fname,
            "pre": pre,
            "post": post,
            "window_size": window_size,
            "median_ll_diff": median_ll
        })

    return pd.DataFrame.from_records(records)

# === Run for both sets ===
df_self = compute_median_ll_diff(self_files, label="self")
df_other = compute_median_ll_diff(other_files, label="other")

# === Combine and view ===
df_combined = pd.concat([df_self, df_other], ignore_index=True)
print(df_combined.sort_values(["source", "window_size"]))


# Visualizations

In [ ]:
import pandas as pd
import glob

# Adapt to your filenames or use Pathlib if preferred
results_dir = "/Users/aniluchavez/Documents/Language/Python/resultsyfi"
all_files = glob.glob(f"{results_dir}/regression_results_PTYFI_task81_*.csv")
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
sns.boxplot(data=df_all, x="window_config", y="ll_diff", hue="condition")
plt.xticks(rotation=45)
plt.title("Log-likelihood Improvement (ll_diff) by Window Sweep and Condition")
plt.axhline(0, linestyle="--", color="gray")
plt.tight_layout()
plt.show()


In [ ]:
summary_stats = (
    df_all.groupby(["window_config", "condition"])["ll_diff"]
    .median()
    .reset_index()
)

# Get the best (highest median) ll_diff for each condition
best_per_condition = summary_stats.loc[
    summary_stats.groupby("condition")["ll_diff"].idxmax()
].sort_values("condition")

# Display result
print("📊 Best-performing window per condition:\n")
print(best_per_condition)


summary_stats = (
    df_all.groupby(["window_config", "condition"])["ll_diff"]
    .median()
    .reset_index()
)

# === Sort by ascending ll_diff for each condition ===
ranked_per_condition = summary_stats.sort_values(
    by=["condition", "ll_diff"],
    ascending=[True, True]
)

# === Print results ===
print("📊 Ranked window configs by median ll_diff (lowest → highest):\n")
print(ranked_per_condition)

 #### window_sweeps = {
    # === Self sweeps: varying Speaker1, others fixed at post-80 ===
     "self_pre_0": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 0},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
     "self_pre_300": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 300},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_pre_500": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 500},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_pre_1000": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 1000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_pre_2000": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 2000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_pre_4000": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 4000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_pre_6000": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 6000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
    "self_onset_300": {
        "Speaker1": {"mode": "onset_to_offset_plus", "post": 300},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },

      "self_onset_500": {
        "Speaker1": {"mode": "onset_to_offset_plus", "post": 500},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },

    "self_onset_1000": {
        "Speaker1": {"mode": "onset_to_offset_plus", "post": 1000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
  
    # === Other sweeps: fixed Speaker1, vary post-onset for others ===
    "other_0": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 2000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 0} for i in range(2, 9)}
    },
    "other_40": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 2000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 40} for i in range(2, 9)}
    },
    "other_80": {
        "Speaker1": {"mode": "pre_to_offset", "pre": 2000},
        **{f"Speaker{i}": {"mode": "onset_to_offset_plus", "post": 80} for i in range(2, 9)}
    },
}